# 40 - speculative decoding and Autopilot on the T4 (Build Guide Phases 11-12)
Run on a T4 x2 session with Internet on and Persistence = Variables and Files. Same setup as notebook 10; benchmark the commit tagged `m3-rc` unless you deliberately move the tag.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must run before torch is imported

In [ ]:
!nvidia-smi
!pip uninstall -y -q torchvision torchaudio
!pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q "transformers>=4.45,<5" safetensors fastapi "uvicorn[standard]" httpx openai \
    prometheus-client psutil hypothesis pytest pytest-asyncio pytest-timeout datasets pandas matplotlib

**Run -> Restart session**, then run the first cell again and continue below.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must run before torch is imported

In [ ]:
!git clone https://github.com/Muskanpandey0304/turingserve.git /kaggle/working/turingserve
%cd /kaggle/working/turingserve
!git checkout m3-rc
!pip install -q -e . --no-deps
!python scripts/check_versions.py

In [ ]:
!hf download Qwen/Qwen2.5-1.5B-Instruct --local-dir /kaggle/working/models/Qwen2.5-1.5B-Instruct
!hf download Qwen/Qwen2.5-0.5B-Instruct --local-dir /kaggle/working/models/Qwen2.5-0.5B-Instruct
!hf download anon8231489123/ShareGPT_Vicuna_unfiltered ShareGPT_V3_unfiltered_cleaned_split.json --repo-type dataset --local-dir /kaggle/working/datasets/sharegpt

In [ ]:
os.environ["TS_MODEL_15"] = "/kaggle/working/models/Qwen2.5-1.5B-Instruct"
os.environ["TS_MODEL_05"] = "/kaggle/working/models/Qwen2.5-0.5B-Instruct"
os.environ["TS_SHAREGPT"] = "/kaggle/working/datasets/sharegpt/ShareGPT_V3_unfiltered_cleaned_split.json"
os.environ["TS_KAGGLE"] = "1"

## Helpers
`start(*extra)` launches a TuringServe server on GPU 0 and waits for /health (prints the log tail if it never comes up); `stop()` terminates it and waits for GPU memory to free.

In [ ]:
import subprocess, time, requests
srv = None

def start(*extra, log="server.log"):
    global srv
    f = open(f"/kaggle/working/{log}", "w")
    srv = subprocess.Popen(["turingserve", "serve", "--model", os.environ["TS_MODEL_15"], "--served-model-name", "qwen",
                            "--port", "8000", *extra], stdout=f, stderr=subprocess.STDOUT)
    for _ in range(240):
        try:
            if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
                print("server up:", extra)
                return
        except Exception:
            pass
        if srv.poll() is not None:
            break
        time.sleep(5)
    print(open(f"/kaggle/working/{log}").read()[-3000:])
    raise RuntimeError("server did not come up")

def stop():
    global srv
    if srv is not None:
        srv.terminate(); srv.wait(); srv = None
    time.sleep(8)


## Smoke test first (about 3 minutes)
Confirms each speculative method starts and serves before the long sweep. If one fails, the log tail is printed above the error; paste it and stop.

In [ ]:
for extra in [("--spec-method", "ngram", "--spec-k", "4"),
              ("--spec-method", "draft", "--spec-k", "4", "--spec-draft-model", os.environ["TS_MODEL_05"])]:
    start(*extra)
    r = requests.post("http://localhost:8000/v1/completions", json={"model": "qwen", "prompt": "The capital of France is", "max_tokens": 16, "temperature": 0})
    print(r.json()["choices"][0]["text"])
    stop()


## Speculative decoding sweep (Build Guide Phase 11, step 4)
ShareGPT, 40 prompts, 2 reps per level, concurrency 1/4/8/16, methods none / ngram k=2,4,6 / draft k=2,4,6 (0.5B draft). About 10 minutes per method, about 70 minutes in total. Each result JSON lands in `/kaggle/working/results/` named by the label. If the session is short on quota, lower `PROMPTS` or drop k=6.

In [ ]:
def bench(*args):
    r = subprocess.run(["python", "-m", "benchmarks.serve_bench", "--base-url", "http://localhost:8000", "--model", "qwen",
                        "--tokenizer", os.environ["TS_MODEL_15"], "--dataset", "sharegpt", "--ignore-eos",
                        "--out", "/kaggle/working/results", *args], capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise RuntimeError("serve_bench failed")

PROMPTS, REPS, LEVELS = 40, 2, "1,4,8,16"
methods = [("none", [])]
methods += [(f"ngram-k{k}", ["--spec-method", "ngram", "--spec-k", str(k)]) for k in (2, 4, 6)]
methods += [(f"draft-k{k}", ["--spec-method", "draft", "--spec-k", str(k), "--spec-draft-model", os.environ["TS_MODEL_05"]]) for k in (2, 4, 6)]
for name, extra in methods:
    start(*extra, log=f"server_spec_{name}.log")
    bench("--num-prompts", str(PROMPTS), "--mode", "concurrency", "--concurrency", LEVELS, "--repeat", str(REPS), "--label", f"spec-{name}")
    stop()


Download every `*_spec-*.json` from `/kaggle/working/results/` (Output panel) and send them over. Then run the Autopilot experiment below, or stop the session to save quota and do it in a second session.

## Autopilot (Build Guide Phase 12, step 3)
Poisson load that goes 0.5, 4, then 0.5 req/s for 120 s each, three configurations: (a) defaults, autopilot off; (b) best fixed configuration from your sweep, autopilot off; (c) `--autopilot full --spec-method ngram`. Edit `BEST_FIXED` for (b) after looking at the sweep (for example a different `--max-num-seqs`); until then (b) reuses the defaults plus ngram k=4, which is a legitimate fixed configuration.

In [ ]:
BEST_FIXED = ["--spec-method", "ngram", "--spec-k", "4"]
runs = [("autopilot-off-default", ["--autopilot", "off"]),
        ("autopilot-off-best-fixed", ["--autopilot", "off", *BEST_FIXED]),
        ("autopilot-full", ["--autopilot", "full", "--spec-method", "ngram"])]
for name, extra in runs:
    start(*extra, log=f"server_{name}.log")
    bench("--num-prompts", "400", "--mode", "profile", "--profile", "120:0.5,120:4,120:0.5", "--label", name)
    stop()


Download the three `*_autopilot-*.json` result files plus the three `server_autopilot-*.log` files (the Autopilot decision log is inside the server log, as JSON lines).